# v0.24d — Frozen Snapshot External Triangulation (Defense / Colab Mirror)

This notebook is a **reproducibility and defense mirror** of the canonical GitHub Actions experiment.

Scientific rules:
- no refit or threshold retuning of the v0.24b champions;
- exact artifact SHA-256 and dependency versions are verified before model loading;
- archived validation replay must reproduce the frozen selected counts exactly;
- external results do **not** authorize LIVE execution;
- the canonical evidence remains the GitHub Actions artifact produced from a frozen source commit.

The notebook intentionally asks you to upload the trusted v0.24b artifact ZIP manually rather than storing any GitHub token or private credential in the notebook.


In [ ]:
# 1) Clone the exact research branch
!rm -rf modular-crypto-trading-bot
!git clone --depth 1 --branch research/v24d-frozen-snapshot-external \
  https://github.com/parsa314/modular-crypto-trading-bot.git
%cd modular-crypto-trading-bot
!git rev-parse HEAD


In [ ]:
# 2) Install the exact model-persistence environment recorded by v0.24b
!python -m pip install -q -e '.[dev]'
!python -m pip install -q --upgrade --force-reinstall \
  'numpy==2.5.3' 'pandas==3.0.5' 'scikit-learn==1.9.1' \
  'joblib==1.6.0' 'scipy==1.18.1' 'threadpoolctl==3.6.0' 'cloudpickle==3.1.2'
!python -m pip check


## Upload the frozen v0.24b artifact

Download `v24b-family-portfolio-34578494059` from GitHub Actions first, then upload that ZIP here.

Expected artifact identity:
- workflow run: `34578494059`
- artifact ID: `10190676664`
- artifact digest recorded by GitHub: `sha256:0384391db85922309e7b67e0e0481f2cbf8795cf069ee48f146e36ba857525db`

The internal model/dataset files are also checked by SHA-256 before deserialization.


In [ ]:
# 3) Upload and extract the trusted artifact; no token is stored in Colab
from google.colab import files
import io, zipfile, pathlib, shutil

uploaded = files.upload()
if len(uploaded) != 1:
    raise RuntimeError("Upload exactly one v24b artifact ZIP.")
name, blob = next(iter(uploaded.items()))
root = pathlib.Path("frozen/v24b")
shutil.rmtree(root, ignore_errors=True)
root.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(blob)) as zf:
    zf.extractall(root)
print("Extracted", name, "to", root)
print(sorted(p.name for p in root.iterdir()))


In [ ]:
# 4) Exact byte/environment checks + archived validation replay
from research_bot.frozen_snapshot_v24d import (
    FrozenSnapshotContract,
    assert_runtime_compatible,
    verify_snapshot_files,
    load_exact_frozen_bundles,
    replay_archived_validation,
)

contract = FrozenSnapshotContract()
print(assert_runtime_compatible(contract))
print(verify_snapshot_files("frozen/v24b", contract))
bundles = load_exact_frozen_bundles("frozen/v24b", contract)
replay = replay_archived_validation("frozen/v24b", bundles, contract)
display(replay)
assert set(replay["status"]) == {"EXACT_FROZEN_BINARY_REPLAY_PASS"}


In [ ]:
# 5) Run deterministic regression guards
!pytest -q tests/test_frozen_snapshot_v24d.py \
  tests/test_v24c_external_mtm_temporal.py \
  tests/test_strategy_family_meta_v24b.py


## Optional external triangulation

The next cell queries public OKX and KuCoin OHLCV through CCXT. Public-exchange availability can differ by network/region. A transport failure is recorded as `BLOCKED`/`DATA_INSUFFICIENT`; it is **not** silently converted into strategy failure.

Even if both venues pass the economic gate, the result is only pre-future-time external evidence. Future-time Plan B, CPCV/PBO/DSR and prospective PAPER evidence remain required.


In [ ]:
# 6) Execute the same v0.24d experiment entry point used by CI
!python scripts/run_v24d_frozen_snapshot_external.py \
  --snapshot-dir frozen/v24b \
  --output-dir artifacts/v24d-colab


In [ ]:
# 7) Defense-readable decision summary
import json, pathlib, pandas as pd
decision_path = pathlib.Path("artifacts/v24d-colab/decision.json")
decision = json.loads(decision_path.read_text())
print(json.dumps(decision, indent=2))

print("\nLIVE:", decision.get("live_execution_authorized"))
print("Forward PAPER:", decision.get("forward_paper_authorized"))
print("Scientific state:", decision.get("state"))
print("Scientific label:", decision.get("scientific_label"))
